# Titanic: train ve test ile tahmin

Bu notebook, ekran görüntüsündeki Kaggle dosyalarını okur.

- `train.csv`: 891 yolcu, `Survived` etiketi var
- `test.csv`: 418 yolcu, etiket yok

Kadın yaşar / erkek ölür kuralı train üzerinde **%78.68** doğruluk verir. Aynı dosyada 5 katmanlı cross-validation, soft vote modelini **%84.40** out-of-fold doğrulukla seçer. Seçilen model tüm train ile yeniden eğitilir ve `output/submission.csv` yazılır.

Kaggle’a bu dosyayı yükle. Public skor, cross-validation sonucunun bir iki puan altında kalabilir.

In [1]:
from pathlib import Path
import sys

import pandas as pd

for candidate in [Path.cwd(), Path.cwd() / "titanic"]:
    if (candidate / "pipeline.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import pipeline

train, test = pipeline.load_data()
print(f"train {train.shape}  test {test.shape}")
train.head()

train (891, 12)  test (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## Özellikler

Model ham isim ve kabin metnini görmez. Satırdan şu alanlar üretilir:

- **Title** — `Master` erkek çocukları `Mr` grubundan ayırır
- **FamilySize / IsAlone** — 2–4 kişilik aileler daha sık hayatta kalır
- **TicketGroup / FarePerPerson** — aynı bileti paylaşanlar ve bilet ücretinin kişi başı hali
- **Age, AgeBand, IsChild** — eksik yaş, aynı ünvan ve sınıfın medyanı ile dolar
- **Deck / HasCabin** — kabin harfi; boş kabin ayrı kategori olarak kalır
- **Sex, Pclass, Embarked**

Yaş, ücret ve liman istatistikleri her cross-validation katında yalnızca o katmanın train parçasına fit edilir. Bilet sayısı etiketsizdir; train ve test birlikte sayılır.

In [3]:
pipeline.preview_features(train, test)

,Age,LogFare,FarePerPerson,FamilySize,SibSp,Parch,TicketGroup,Pclass,IsAlone,HasCabin,IsChild,Sex,Embarked,Title,Deck,AgeBand
0,22.0,2.110213,7.25000,2.0,1.0,0.0,1.0,3.0,0.0,0.0,0.0,male,S,Mr,U,Young
1,38.0,4.280593,35.64165,2.0,1.0,0.0,2.0,1.0,0.0,1.0,0.0,female,C,Mrs,C,Adult
2,26.0,2.188856,7.92500,1.0,0.0,0.0,1.0,3.0,1.0,0.0,0.0,female,S,Miss,U,Young
3,35.0,3.990834,26.55000,2.0,1.0,0.0,2.0,1.0,0.0,1.0,0.0,female,S,Mrs,C,Young
4,35.0,2.202765,8.05000,1.0,0.0,0.0,1.0,3.0,1.0,0.0,0.0,male,S,Mr,U,Young


## Model seçimi

Adaylar: lojistik regresyon, random forest, gradient boosting, histogram gradient boosting ve bunların soft vote birleşimi. Metrik accuracy. En yüksek ortalama doğruluğu veren model gönderiyi üretir.

In [4]:
report = pipeline.run()
report.scores

Kadın yaşar / erkek ölür kuralı (train): 78.68%

5-kat çapraz doğrulama
                 model mean_accuracy std_accuracy
             soft_vote        84.40%        2.03%
     gradient_boosting        83.95%        1.99%
hist_gradient_boosting        83.72%        1.28%
         random_forest        83.28%        1.24%
              logistic        82.38%        1.31%

Seçilen model: soft_vote
Out-of-fold doğruluk: 84.40%

Cinsiyet ve sınıfa göre out-of-fold doğruluk
   Sex  Pclass   n accuracy
female       1  94   96.81%
female       2  76   92.11%
female       3 144   69.44%
  male       1 122   65.57%
  male       2 108   92.59%
  male       3 347   89.63%

Gönderi: /workspace/titanic/output/submission.csv (418 satır)


,model,mean_accuracy,std_accuracy,fold_1,fold_2,fold_3,fold_4,fold_5
0,soft_vote,0.843965,0.020265,0.871508,0.842697,0.814607,0.842697,0.848315
1,gradient_boosting,0.839483,0.019906,0.860335,0.831461,0.814607,0.831461,0.859551
2,hist_gradient_boosting,0.837242,0.012838,0.854749,0.831461,0.820225,0.837079,0.842697
3,random_forest,0.832760,0.012367,0.843575,0.814607,0.825843,0.837079,0.842697
4,logistic,0.823784,0.013083,0.832402,0.820225,0.803371,0.825843,0.837079


In [5]:
report.oof_by_group

,Sex,Pclass,n,accuracy
0,female,1,94,0.968085
1,female,2,76,0.921053
2,female,3,144,0.694444
3,male,1,122,0.655738
4,male,2,108,0.925926
5,male,3,347,0.896254


Kalan hatalar iki grupta toplanır: 3. sınıf kadınlar (%69.44) ve 1. sınıf erkekler (%65.57). 2. ve 3. sınıf erkeklerde doğruluk yüksektir; bu grupların çoğu ölür ve model de ölümü seçer.

Test tahminlerinde hayatta kalma oranı %38.0’dır (train %38.4). Model, kadın/erkek kuralından 45 satırda ayrılır: bazı 3. sınıf kadınları kaybedilmiş, `Master` ünvanlı erkek çocukları ve bir kısım 1. sınıf erkekleri kurtulmuş sayar. Ekran görüntüsündeki ilk test satırlarında Kelly, Myles ve Wirz için 0; Hirvonen için 1 üretir.

## Gönderi

`Survived` burada tahmindir. Test etiketleri dosyada yoktur. Kaggle’a `titanic/output/submission.csv` yüklenir.

In [6]:
report.submission.head(10)

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
